# 유치원 앱 수익화 비판적 재검토

Mixpanel, App Store Connect, 공식 공시·후기·결원 데이터의 최신 스냅샷을 결합해 수익화 의사결정을 재현한다. 관측값과 가정값을 분리하며, 결과는 `analysis-output.json`으로 저장한다.

In [1]:
from pathlib import Path
import json
import math
import pandas as pd

cwd = Path.cwd()
root = next(candidate for candidate in [cwd, *cwd.parents] if (candidate / 'package.json').exists())
report_dir = root / 'docs/reports/kindergarten-retention-monetization-2026-08-06/critical-review'
inputs = json.loads((report_dir / 'analysis-input.json').read_text())
kindergarten_meta = json.loads((root / 'public/data/kindergartens.meta.json').read_text())
review_meta = json.loads((root / 'public/data/reviews.meta.json').read_text())
reviews = json.loads((root / 'public/data/reviews.json').read_text())
region_meta = json.loads((root / 'public/data/region-codes.meta.json').read_text())
vacancy = json.loads((root / 'public/data/vacancy.json').read_text())
print({'snapshot': inputs['snapshot_date'], 'rolling_30d_active': inputs['mixpanel']['rolling_30_day']['active_users']})

{'snapshot': '2026-08-06', 'rolling_30d_active': 186}


## 행동과 리텐션

행동 도달률은 동일한 124명 분모의 1일 퍼널이다. 리텐션은 해당 시점까지 성숙한 코호트의 분자·분모를 합산해 다시 계산한다.

In [2]:
behavior = pd.DataFrame(inputs['mixpanel']['one_day_behavior_funnels'])
behavior['recomputed_rate'] = behavior['converted_users'] / behavior['denominator_users']
assert (behavior['recomputed_rate'] - behavior['conversion_rate']).abs().max() < 1e-6
retention = pd.DataFrame(inputs['mixpanel']['new_user_retention_30d_window'])
retention['recomputed_rate'] = retention['returned_users'] / retention['eligible_users']
assert (retention['recomputed_rate'] - retention['rate']).abs().max() < 1e-6
display(behavior[['behavior', 'converted_users', 'denominator_users', 'conversion_rate']])
display(retention[['day', 'returned_users', 'eligible_users', 'rate']])

,behavior,converted_users,denominator_users,conversion_rate
0,후보 2곳 도달,47,124,0.379032
1,2곳 이상 실제 비교 보기,42,124,0.338710
2,외부 후기 열기,31,124,0.250000
3,즐겨찾기 2곳 도달,17,124,0.137097
4,비교 공유 시작,3,124,0.024194


,day,returned_users,eligible_users,rate
0,D1,4,168,0.023810
1,D7,5,132,0.037879
2,D14,1,31,0.032258
3,D28,0,8,0.000000


## ASC와 Mixpanel 신규 신호 정합성

두 지표는 날짜 기준과 정의가 달라 직접 합산하지 않는다. 시차별 피어슨 상관만 진단해 유입 스파이크가 같은 현상을 반영하는지 확인한다.

In [3]:
asc = pd.Series(inputs['app_store_connect']['daily'], dtype='float64')
asc.index = pd.to_datetime(asc.index)
mix_new = pd.Series(inputs['mixpanel']['daily_new_launch_users'], dtype='float64')
mix_new.index = pd.to_datetime(mix_new.index)
lag_rows = []
for lag in range(-2, 3):
    shifted = asc.copy()
    shifted.index = shifted.index + pd.Timedelta(days=lag)
    aligned = pd.concat([shifted.rename('asc'), mix_new.rename('mixpanel')], axis=1).dropna()
    lag_rows.append({'asc_date_shift_days': lag, 'matched_days': len(aligned), 'pearson_r': aligned.corr().iloc[0, 1]})
lag_diagnostics = pd.DataFrame(lag_rows)
display(lag_diagnostics)
print({'asc_known_units': int(asc.sum()), 'mixpanel_new_launch_sum': int(mix_new.sum())})

,asc_date_shift_days,matched_days,pearson_r
0,-2,17,0.092975
1,-1,18,0.714678
2,0,19,0.615671
3,1,19,0.346743
4,2,18,0.077581


{'asc_known_units': 106, 'mixpanel_new_launch_sum': 170}


## B2C 민감도

활성 사용자 186명은 실제 30일 고유 사용자다. 결제율은 관측되지 않은 가정이므로 가격별 민감도만 계산한다.

In [4]:
active_users = inputs['mixpanel']['rolling_30_day']['active_users']
eligible_rate = inputs['economics']['eligible_rate']
target = inputs['economics']['gross_monthly_target_krw']
economics_rows = []
for price in inputs['economics']['prices_krw']:
    for purchase_rate in inputs['economics']['eligible_purchase_rates']:
        purchases = active_users * eligible_rate * purchase_rate
        economics_rows.append({
            'price_krw': price,
            'eligible_purchase_rate': purchase_rate,
            'estimated_purchases': round(purchases, 1),
            'gross_revenue_krw': round(purchases * price),
            'active_users_for_1m': round(target / (price * purchase_rate * eligible_rate)),
        })
economics = pd.DataFrame(economics_rows)
display(economics)

,price_krw,eligible_purchase_rate,estimated_purchases,gross_revenue_krw,active_users_for_1m
0,9900,0.03,2.1,20938,8883
1,9900,0.05,3.5,34897,5330
2,9900,0.10,7.0,69795,2665
3,19900,0.03,2.1,42088,4419
4,19900,0.05,3.5,70147,2652
5,19900,0.10,7.0,140295,1326


## 공개 데이터 품질과 결과 저장

In [5]:
quality = {
    'kindergartens': {
        'source_version': kindergarten_meta['sourceVersion'],
        'collected_at': kindergarten_meta['collectedAt'],
        'total_count': kindergarten_meta['totalCount'],
        'registry_join_coverage': kindergarten_meta['registryJoinCoverage'],
    },
    'reviews': {
        'version': reviews['version'],
        'generated_at': review_meta['generatedAt'],
        'total_count': reviews['totalCount'],
        'kindergarten_count': reviews['kindergartenCount'],
        'catalog_coverage': review_meta['coverageRate'],
        'excluded_orphan_reviews': review_meta['excludedOrphanReviewCount'],
        'deduplicated_reviews': review_meta['duplicateReviewCount'],
    },
    'vacancy': {
        'version': vacancy['version'],
        'total_count': vacancy['totalCount'],
        'positive_count': vacancy['positiveCount'],
        'quality': vacancy.get('quality'),
    },
    'region_codes': {
        'checked_at': region_meta['checkedAt'],
        'total_count': region_meta['totalCount'],
    },
}
assert quality['kindergartens']['total_count'] == 7152
assert quality['reviews']['total_count'] == 6055

output = {
    'snapshot_date': inputs['snapshot_date'],
    'complete_data_through': inputs['complete_data_through'],
    'behavior_signals': behavior.to_dict(orient='records'),
    'new_user_retention': retention.to_dict(orient='records'),
    'two_candidate_retention': inputs['mixpanel']['two_candidate_retention'],
    'acquisition_alignment': {
        'asc_known_units': int(asc.sum()),
        'mixpanel_new_launch_sum': int(mix_new.sum()),
        'known_missing_asc_dates': inputs['app_store_connect']['known_missing_dates'],
        'lag_diagnostics': lag_diagnostics.round(6).to_dict(orient='records'),
    },
    'economics_scenarios': economics.to_dict(orient='records'),
    'public_data_quality': quality,
}
(report_dir / 'analysis-output.json').write_text(json.dumps(output, ensure_ascii=False, indent=2))
print(report_dir / 'analysis-output.json')

/Users/solkim/Dev/where_kindergarden/docs/reports/kindergarten-retention-monetization-2026-08-06/critical-review/analysis-output.json
